# eiapy Showcase

Comprehensive demonstration of **eiapy** -- a Python client for the U.S. EIA Open Data API v2.

This notebook covers all **13 energy categories** using both the generic `get_data()` API and the natural-gas convenience functions.

**Prerequisites:**
```bash
export EIA_API_KEY="your_key_here"
pip install eiapy
```

In [1]:
from eiapy import (
    get_data,
    get_metadata,
    list_categories,
    list_groups,
    # Natural gas convenience
    get_consumption,
    get_exploration,
    get_movements,
    get_prices,
    get_production,
    get_storage,
    get_summary,
    get_available_series,
    # Route helpers
    list_routes,
    resolve_route,
)

---

## 0. Discovery -- Explore What's Available

Before querying data, use the discovery functions to see all categories, groups, and available facets.

In [2]:
# All 13 EIA categories
for cat in list_categories():
    groups = list_groups(cat)
    total_variants = sum(len(v) for v in groups.values())
    print(f"  {cat:25s} -- {len(groups)} groups, {total_variants} variants")

  aeo                       -- 8 groups, 8 variants
  coal                      -- 8 groups, 13 variants
  crude-oil-imports         -- 1 groups, 1 variants
  densified-biomass         -- 8 groups, 8 variants
  electricity               -- 6 groups, 19 variants
  ieo                       -- 1 groups, 1 variants
  international             -- 1 groups, 1 variants
  natural-gas               -- 7 groups, 53 variants
  nuclear-outages           -- 3 groups, 3 variants
  petroleum                 -- 7 groups, 112 variants
  seds                      -- 1 groups, 1 variants
  steo                      -- 1 groups, 1 variants
  total-energy              -- 1 groups, 1 variants


In [3]:
# Drill into a specific category
list_groups("electricity")

{'retail-sales': ['retail-sales'],
 'electric-power-operational-data': ['electric-power-operational-data'],
 'rto': ['daily-fuel-type-data',
  'daily-interchange-data',
  'daily-region-data',
  'daily-region-sub-ba-data',
  'fuel-type-data',
  'interchange-data',
  'region-data',
  'region-sub-ba-data'],
 'state-electricity-profiles': ['capability',
  'emissions-by-state-by-fuel',
  'energy-efficiency',
  'meters',
  'net-metering',
  'source-disposition',
  'summary'],
 'operating-generator-capacity': ['operating-generator-capacity'],
 'facility-fuel': ['facility-fuel']}

In [4]:
# Natural gas route discovery (legacy helper)
list_routes()

{'consumption': ['acct', 'heat', 'num', 'pns', 'sum'],
 'movements': ['expc', 'impc', 'ist', 'poe1', 'poe2', 'state'],
 'production': ['coalbed',
  'deep',
  'lc',
  'ngpl',
  'off',
  'oilwells',
  'pp',
  'shalegas',
  'ss',
  'sum',
  'wells',
  'whv'],
 'storage': ['cap', 'lng', 'sum', 'type', 'wkly'],
 'prices': ['fut', 'rescom', 'sum'],
 'summary': ['lsum', 'snd', 'sndm'],
 'exploration': ['adng',
  'coalbed',
  'cplc',
  'deep',
  'drill',
  'dry',
  'lc',
  'nang',
  'ngl',
  'ngpl',
  'nprod',
  'seis',
  'shalegas',
  'sum',
  'wals',
  'wellcost',
  'welldep',
  'wellend',
  'wellfoot']}

In [5]:
# Inspect metadata for any endpoint -- discover facets, frequencies, data columns
meta = get_metadata("electricity", "retail-sales")
print(f"Frequencies: {[f['id'] for f in meta.get('frequency', [])]}")
print(f"Facets:      {[f['id'] for f in meta.get('facets', [])]}")
print(f"Data cols:   {list(meta.get('data', {}).keys())}")

Frequencies: ['monthly', 'quarterly', 'annual']
Facets:      ['stateid', 'sectorid']
Data cols:   ['revenue', 'sales', 'price', 'customers']


---

## 1. Electricity

### 1a. Retail Sales

In [6]:
# Monthly residential electricity in Texas, 2023-2024
df = get_data(
    "electricity", "retail-sales",
    facets={"stateid": "TX", "sectorid": "RES"},
    data=["revenue", "sales", "price", "customers"],
    frequency="monthly",
    start="2023-01",
    end="2024-12",
)
df.head(10)

,period,stateid,stateDescription,sectorid,sectorName,revenue,sales,price,customers,revenue-units,sales-units,price-units,customers-units
0,2024-12-01,TX,Texas,RES,residential,1699.50026,11067.5171,15.36,12658594,million dollars,million kilowatt hours,cents per kilowatt-hour,number of customers
1,2024-11-01,TX,Texas,RES,residential,1634.97778,10425.46816,15.68,12586383,million dollars,million kilowatt hours,cents per kilowatt-hour,number of customers
2,2024-10-01,TX,Texas,RES,residential,2122.76746,13543.09978,15.67,12649326,million dollars,million kilowatt hours,cents per kilowatt-hour,number of customers
3,2024-09-01,TX,Texas,RES,residential,2460.5289,16336.6695,15.06,12637695,million dollars,million kilowatt hours,cents per kilowatt-hour,number of customers
4,2024-08-01,TX,Texas,RES,residential,2958.62278,19878.28376,14.88,12658469,million dollars,million kilowatt hours,cents per kilowatt-hour,number of customers
5,2024-07-01,TX,Texas,RES,residential,2773.63449,18865.604,14.7,12636608,million dollars,million kilowatt hours,cents per kilowatt-hour,number of customers
6,2024-06-01,TX,Texas,RES,residential,2490.44615,17114.45145,14.55,12574017,million dollars,million kilowatt hours,cents per kilowatt-hour,number of customers
7,2024-05-01,TX,Texas,RES,residential,1895.60149,12784.10366,14.83,12534929,million dollars,million kilowatt hours,cents per kilowatt-hour,number of customers
8,2024-04-01,TX,Texas,RES,residential,1476.23282,9754.68291,15.13,12485426,million dollars,million kilowatt hours,cents per kilowatt-hour,number of customers
9,2024-03-01,TX,Texas,RES,residential,1419.20682,9450.32398,15.02,12407521,million dollars,million kilowatt hours,cents per kilowatt-hour,number of customers


In [7]:
# Annual electricity sales across multiple states, all sectors
df = get_data(
    "electricity", "retail-sales",
    facets={"stateid": ["CA", "NY", "FL", "IL"]},
    data=["sales", "price"],
    frequency="annual",
    start="2018",
    end="2023",
)
df.head(10)

,period,stateid,stateDescription,sectorid,sectorName,sales,price,sales-units,price-units
0,2023-01-01,CA,California,ALL,all sectors,239480.45202,24.87,million kilowatt hours,cents per kilowatt-hour
1,2023-01-01,CA,California,COM,commercial,112935.56601,23.91,million kilowatt hours,cents per kilowatt-hour
2,2023-01-01,CA,California,IND,industrial,42998.55699,18.64,million kilowatt hours,cents per kilowatt-hour
3,2023-01-01,CA,California,OTH,other,NaN,NaN,million kilowatt hours,cents per kilowatt-hour
4,2023-01-01,CA,California,RES,residential,82820.89801,29.51,million kilowatt hours,cents per kilowatt-hour
5,2023-01-01,CA,California,TRA,transportation,725.43101,13.01,million kilowatt hours,cents per kilowatt-hour
6,2023-01-01,FL,Florida,ALL,all sectors,250940.214,13.53,million kilowatt hours,cents per kilowatt-hour
7,2023-01-01,FL,Florida,COM,commercial,97255.44301,11.95,million kilowatt hours,cents per kilowatt-hour
8,2023-01-01,FL,Florida,IND,industrial,17808.644,9.39,million kilowatt hours,cents per kilowatt-hour
9,2023-01-01,FL,Florida,OTH,other,NaN,NaN,million kilowatt hours,cents per kilowatt-hour


In [8]:
# Quarterly commercial electricity for the whole US
df = get_data(
    "electricity", "retail-sales",
    facets={"sectorid": "COM"},
    data=["sales", "revenue"],
    frequency="quarterly",
    start="2022-01-02",
)
df.head()

/Users/egutierrez/Documents/code/pygasflow/eiapy/client.py:122: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["period"] = pd.to_datetime(df["period"])


,period,stateid,stateDescription,sectorid,sectorName,sales,revenue,sales-units,revenue-units
0,2025-10-01,AK,Alaska,COM,commercial,689.05371,151.12468,million kilowatt hours,million dollars
1,2025-10-01,AL,Alabama,COM,commercial,5340.41191,783.63223,million kilowatt hours,million dollars
2,2025-10-01,AR,Arkansas,COM,commercial,2752.27677,298.80949,million kilowatt hours,million dollars
3,2025-10-01,AZ,Arizona,COM,commercial,8944.80633,1064.90577,million kilowatt hours,million dollars
4,2025-10-01,CA,California,COM,commercial,28801.42266,7529.06727,million kilowatt hours,million dollars


### 1b. Facility Fuel & Generation

In [9]:
# Monthly generation from natural gas plants in Texas
df = get_data(
    "electricity", "facility-fuel",
    facets={"state": "TX", "fuel2002": "NG"},
    data=["generation"],
    frequency="monthly",
    start="2024-01",
    end="2024-06",
)
df.head(10)

,period,plantCode,plantName,fuel2002,fuelTypeDescription,state,stateDescription,primeMover,generation,generation-units
0,2024-06-01,9,Copper,NG,Natural Gas,TX,Texas,ALL,5687.68,megawatthours
1,2024-06-01,9,Copper,NG,Natural Gas,TX,Texas,GT,5687.68,megawatthours
2,2024-06-01,298,Limestone,NG,Natural Gas,TX,Texas,ALL,11395.09,megawatthours
3,2024-06-01,298,Limestone,NG,Natural Gas,TX,Texas,ST,11395.09,megawatthours
4,2024-06-01,3439,Laredo,NG,Natural Gas,TX,Texas,ALL,6682.23,megawatthours
5,2024-06-01,3439,Laredo,NG,Natural Gas,TX,Texas,GT,6682.23,megawatthours
6,2024-06-01,3441,Nueces Bay,NG,Natural Gas,TX,Texas,ALL,196177,megawatthours
7,2024-06-01,3441,Nueces Bay,NG,Natural Gas,TX,Texas,CA,591,megawatthours
8,2024-06-01,3441,Nueces Bay,NG,Natural Gas,TX,Texas,CT,195586,megawatthours
9,2024-06-01,3443,Victoria,NG,Natural Gas,TX,Texas,ALL,172978,megawatthours


### 1c. State Electricity Profiles -- Emissions

In [10]:
# Power sector emissions by fuel for Texas & California
df = get_data(
    "electricity", "state-electricity-profiles",
    variant="emissions-by-state-by-fuel",
    facets={"stateid": ["TX", "CA"]},
    frequency="annual",
    start="2019",
)
df.head(10)

,period,stateid,stateDescription,fuelid,fuelDescription,so2-rate-lbs-mwh,so2-short-tons,nox-rate-lbs-mwh,nox-short-tons,co2-rate-lbs-mwh,co2-thousand-metric-tons,so2-rate-lbs-mwh-units,so2-short-tons-units,nox-rate-lbs-mwh-units,nox-short-tons-units,co2-rate-lbs-mwh-units,co2-thousand-metric-tons-units
0,2024-01-01,CA,California,ALL,Total,0,748,.6,62519,407,39594,pounds per megawatthour,short tons,pounds per megawatthour,short tons,pounds per megawatthour,thousand metric tons
1,2024-01-01,CA,California,COL,Coal,NaN,119,NaN,797,NaN,1129,pounds per megawatthour,short tons,pounds per megawatthour,short tons,pounds per megawatthour,thousand metric tons
2,2024-01-01,CA,California,NG,Natural Gas,NaN,209,NaN,38470,NaN,37951,pounds per megawatthour,short tons,pounds per megawatthour,short tons,pounds per megawatthour,thousand metric tons
3,2024-01-01,CA,California,OTH,Other,NaN,346,NaN,22617,NaN,458,pounds per megawatthour,short tons,pounds per megawatthour,short tons,pounds per megawatthour,thousand metric tons
4,2024-01-01,CA,California,PET,Petroleum,NaN,74,NaN,635,NaN,56,pounds per megawatthour,short tons,pounds per megawatthour,short tons,pounds per megawatthour,thousand metric tons
5,2024-01-01,TX,Texas,ALL,Total,.3,92903,.6,161470,823,211902,pounds per megawatthour,short tons,pounds per megawatthour,short tons,pounds per megawatthour,thousand metric tons
6,2024-01-01,TX,Texas,COL,Coal,NaN,87334,NaN,43971,NaN,72304,pounds per megawatthour,short tons,pounds per megawatthour,short tons,pounds per megawatthour,thousand metric tons
7,2024-01-01,TX,Texas,NG,Natural Gas,NaN,848,NaN,109728,NaN,139307,pounds per megawatthour,short tons,pounds per megawatthour,short tons,pounds per megawatthour,thousand metric tons
8,2024-01-01,TX,Texas,OTH,Other,NaN,4101,NaN,6994,NaN,0,pounds per megawatthour,short tons,pounds per megawatthour,short tons,pounds per megawatthour,thousand metric tons
9,2024-01-01,TX,Texas,PET,Petroleum,NaN,620,NaN,777,NaN,292,pounds per megawatthour,short tons,pounds per megawatthour,short tons,pounds per megawatthour,thousand metric tons


---

## 2. Petroleum

### 2a. Crude Oil Production

In [11]:
# Monthly crude oil production in Texas and North Dakota
df = get_data(
    "petroleum", "crd", "crpdn",
    facets={"duoarea": ["STX", "SND"]},
    frequency="monthly",
    start="2023-01",
    end="2024-06",
)
df.head(10)

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2023-01-01,SND,USA-ND,EPC0,Crude Oil,FPF,Field Production,MCRFPND1,North Dakota Field Production of Crude Oil (Th...,32651,MBBL
1,2023-02-01,SND,USA-ND,EPC0,Crude Oil,FPF,Field Production,MCRFPND1,North Dakota Field Production of Crude Oil (Th...,32128,MBBL
2,2023-03-01,SND,USA-ND,EPC0,Crude Oil,FPF,Field Production,MCRFPND1,North Dakota Field Production of Crude Oil (Th...,34487,MBBL
3,2023-04-01,SND,USA-ND,EPC0,Crude Oil,FPF,Field Production,MCRFPND1,North Dakota Field Production of Crude Oil (Th...,33678,MBBL
4,2023-05-01,SND,USA-ND,EPC0,Crude Oil,FPF,Field Production,MCRFPND1,North Dakota Field Production of Crude Oil (Th...,34936,MBBL
5,2023-06-01,SND,USA-ND,EPC0,Crude Oil,FPF,Field Production,MCRFPND1,North Dakota Field Production of Crude Oil (Th...,34813,MBBL
6,2023-07-01,SND,USA-ND,EPC0,Crude Oil,FPF,Field Production,MCRFPND1,North Dakota Field Production of Crude Oil (Th...,36358,MBBL
7,2023-08-01,SND,USA-ND,EPC0,Crude Oil,FPF,Field Production,MCRFPND1,North Dakota Field Production of Crude Oil (Th...,37412,MBBL
8,2023-09-01,SND,USA-ND,EPC0,Crude Oil,FPF,Field Production,MCRFPND1,North Dakota Field Production of Crude Oil (Th...,38608,MBBL
9,2023-10-01,SND,USA-ND,EPC0,Crude Oil,FPF,Field Production,MCRFPND1,North Dakota Field Production of Crude Oil (Th...,38851,MBBL


### 2b. Motor Gasoline Prices

In [12]:
# Monthly retail gasoline prices (all areas)
df = get_data(
    "petroleum", "pri", "allmg",
    frequency="monthly",
    start="2024-01",
    end="2024-06",
)
df.head(10)

""


### 2c. Weekly Supply & Disposition

In [13]:
# Weekly petroleum supply & disposition
df = get_data(
    "petroleum", "sum", "sndw",
    frequency="weekly",
    start="2024-01",
    end="2024-03",
)
df.head(10)

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2024-01-05,NUS,U.S.,EPDM10,"Distillate Fuel Oil, Greater than 15 to 500 pp...",SAE,Ending Stocks,WD1ST_NUS_1,"U.S. Ending Stocks of Distillate Fuel Oil, Gre...",2267.0,MBBL
1,2024-01-12,NUS,U.S.,EPDM10,"Distillate Fuel Oil, Greater than 15 to 500 pp...",SAE,Ending Stocks,WD1ST_NUS_1,"U.S. Ending Stocks of Distillate Fuel Oil, Gre...",2475.0,MBBL
2,2024-01-19,NUS,U.S.,EPDM10,"Distillate Fuel Oil, Greater than 15 to 500 pp...",SAE,Ending Stocks,WD1ST_NUS_1,"U.S. Ending Stocks of Distillate Fuel Oil, Gre...",2550.0,MBBL
3,2024-01-26,NUS,U.S.,EPDM10,"Distillate Fuel Oil, Greater than 15 to 500 pp...",SAE,Ending Stocks,WD1ST_NUS_1,"U.S. Ending Stocks of Distillate Fuel Oil, Gre...",2264.0,MBBL
4,2024-02-02,NUS,U.S.,EPDM10,"Distillate Fuel Oil, Greater than 15 to 500 pp...",SAE,Ending Stocks,WD1ST_NUS_1,"U.S. Ending Stocks of Distillate Fuel Oil, Gre...",2459.0,MBBL
5,2024-02-09,NUS,U.S.,EPDM10,"Distillate Fuel Oil, Greater than 15 to 500 pp...",SAE,Ending Stocks,WD1ST_NUS_1,"U.S. Ending Stocks of Distillate Fuel Oil, Gre...",2554.0,MBBL
6,2024-02-16,NUS,U.S.,EPDM10,"Distillate Fuel Oil, Greater than 15 to 500 pp...",SAE,Ending Stocks,WD1ST_NUS_1,"U.S. Ending Stocks of Distillate Fuel Oil, Gre...",2646.0,MBBL
7,2024-02-23,NUS,U.S.,EPDM10,"Distillate Fuel Oil, Greater than 15 to 500 pp...",SAE,Ending Stocks,WD1ST_NUS_1,"U.S. Ending Stocks of Distillate Fuel Oil, Gre...",2527.0,MBBL
8,2024-03-01,NUS,U.S.,EPDM10,"Distillate Fuel Oil, Greater than 15 to 500 pp...",SAE,Ending Stocks,WD1ST_NUS_1,"U.S. Ending Stocks of Distillate Fuel Oil, Gre...",2595.0,MBBL
9,2024-01-05,R10,PADD 1,EPDM10,"Distillate Fuel Oil, Greater than 15 to 500 pp...",SAE,Ending Stocks,WD1ST_R10_1,East Coast (PADD 1) Ending Stocks of Distillat...,553.0,MBBL


---

## 3. Coal

### 3a. Consumption & Quality

In [14]:
# Annual coal consumption by power sector in Wyoming & West Virginia
df = get_data(
    "coal", "consumption-and-quality",
    facets={"location": ["WY", "WV"], "sector": ["1"]},
    data=["consumption", "price", "heat-content", "sulfur-content"],
    frequency="annual",
    start="2018",
    end="2023",
)
df.head(10)

,period,location,stateDescription,sector,sectorDescription,consumption,price,heat-content,sulfur-content,consumption-units,price-units,heat-content-units,sulfur-content-units
0,2023-01-01,WV,West Virginia,1,Electric Utility,14874567,74,25.1131,2.86,short tons,dollars per short ton,average sulfur percent by weight,average sulfur percent by weight
1,2023-01-01,WY,Wyoming,1,Electric Utility,19423478,32.43,17.2195,.4,short tons,dollars per short ton,average sulfur percent by weight,average sulfur percent by weight
2,2022-01-01,WV,West Virginia,1,Electric Utility,15659671,61.11,25.0746,2.93,short tons,dollars per short ton,average sulfur percent by weight,average sulfur percent by weight
3,2022-01-01,WY,Wyoming,1,Electric Utility,20324343,28.62,17.4419,.41,short tons,dollars per short ton,average sulfur percent by weight,average sulfur percent by weight
4,2021-01-01,WV,West Virginia,1,Electric Utility,18638734,NaN,25.012,3.07,short tons,dollars per short ton,average sulfur percent by weight,average sulfur percent by weight
5,2021-01-01,WY,Wyoming,1,Electric Utility,19622190,NaN,17.5461,.41,short tons,dollars per short ton,average sulfur percent by weight,average sulfur percent by weight
6,2020-01-01,WV,West Virginia,1,Electric Utility,16264521,NaN,25.118,2.95,short tons,dollars per short ton,average sulfur percent by weight,average sulfur percent by weight
7,2020-01-01,WY,Wyoming,1,Electric Utility,20339363,NaN,17.4906,.42,short tons,dollars per short ton,average sulfur percent by weight,average sulfur percent by weight
8,2019-01-01,WV,West Virginia,1,Electric Utility,19094260,NaN,25.1698,2.87,short tons,dollars per short ton,average sulfur percent by weight,average sulfur percent by weight
9,2019-01-01,WY,Wyoming,1,Electric Utility,21293263,NaN,17.4114,.43,short tons,dollars per short ton,average sulfur percent by weight,average sulfur percent by weight


### 3b. Shipments by Mine State

In [15]:
# Quarterly coal shipments from Pennsylvania, bituminous rank
df = get_data(
    "coal", "shipments", "mine-state-aggregates",
    facets={"mineStateId": "PA", "coalRankId": "BIT"},
    data=["quantity", "price"],
    frequency="quarterly",
    start="2022",
)
df.head(10)

/Users/egutierrez/Documents/code/pygasflow/eiapy/client.py:122: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["period"] = pd.to_datetime(df["period"])


,period,mineStateId,mineStateDescription,coalRankId,coalRankDescription,quantity,price,quantity-units,price-units
0,2025-10-01,PA,Pennsylvania,BIT,Bituminous,4561785,83.09,tons,average dollars per ton
1,2025-07-01,PA,Pennsylvania,BIT,Bituminous,4781647,85.28,tons,average dollars per ton
2,2025-04-01,PA,Pennsylvania,BIT,Bituminous,4590820,86.05,tons,average dollars per ton
3,2025-01-01,PA,Pennsylvania,BIT,Bituminous,3721021,87.78,tons,average dollars per ton
4,2024-10-01,PA,Pennsylvania,BIT,Bituminous,3706336,80.55,tons,average dollars per ton
5,2024-07-01,PA,Pennsylvania,BIT,Bituminous,3717477,81.2,tons,average dollars per ton
6,2024-04-01,PA,Pennsylvania,BIT,Bituminous,3932195,85.95,tons,average dollars per ton
7,2024-01-01,PA,Pennsylvania,BIT,Bituminous,3354759,86.41,tons,average dollars per ton
8,2023-10-01,PA,Pennsylvania,BIT,Bituminous,4210035,88.69,tons,average dollars per ton
9,2023-07-01,PA,Pennsylvania,BIT,Bituminous,4065629,87.39,tons,average dollars per ton


---

## 4. Crude Oil Imports

In [16]:
# Monthly crude imports (all origins)
df = get_data(
    "crude-oil-imports",
    frequency="monthly",
    start="2024-01",
    end="2024-06",
)
df.head(10)

,period,originId,originName,originType,originTypeName,destinationId,destinationName,destinationType,destinationTypeName,gradeId,gradeName,quantity,quantity-units
0,2024-06-01,CTY_AE,United Arab Emirates,CTY,Country,PP_5,PADD5 (West Coast),PP,Port PADD,LSO,Light Sour,1697,thousand barrels
1,2024-06-01,CTY_AE,United Arab Emirates,CTY,Country,PP_5,PADD5 (West Coast),PP,Port PADD,MED,Medium,298,thousand barrels
2,2024-06-01,CTY_AE,United Arab Emirates,CTY,Country,PS_CA,California,PS,Port State,LSO,Light Sour,1697,thousand barrels
3,2024-06-01,CTY_AE,United Arab Emirates,CTY,Country,PS_CA,California,PS,Port State,MED,Medium,298,thousand barrels
4,2024-06-01,CTY_AE,United Arab Emirates,CTY,Country,PT_2812,"Richmond, CA",PT,Port,LSO,Light Sour,1697,thousand barrels
5,2024-06-01,CTY_AE,United Arab Emirates,CTY,Country,PT_2812,"Richmond, CA",PT,Port,MED,Medium,298,thousand barrels
6,2024-06-01,CTY_AE,United Arab Emirates,CTY,Country,RF_120,CHEVRON USA / RICHMOND / CA,RF,Refinery,LSO,Light Sour,1697,thousand barrels
7,2024-06-01,CTY_AE,United Arab Emirates,CTY,Country,RF_120,CHEVRON USA / RICHMOND / CA,RF,Refinery,MED,Medium,298,thousand barrels
8,2024-06-01,CTY_AE,United Arab Emirates,CTY,Country,RP_5,PADD5 (West Coast),RP,Refinery PADD,LSO,Light Sour,1697,thousand barrels
9,2024-06-01,CTY_AE,United Arab Emirates,CTY,Country,RP_5,PADD5 (West Coast),RP,Refinery PADD,MED,Medium,298,thousand barrels


In [17]:
# Annual crude imports by grade -- heavy vs. light
df = get_data(
    "crude-oil-imports",
    facets={"gradeId": ["H", "L"]},
    frequency="annual",
    start="2020",
    end="2023",
)
df.head(10)

""


---

## 5. Natural Gas -- Convenience Functions

These provide friendly parameter names (`state=`, `region=`, `area=`) with automatic facet mapping.

### 5a. Consumption

In [18]:
# Monthly Texas consumption
df = get_consumption(state="TX", frequency="monthly", start="2023-01", end="2024-06")
df.head(10)

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2023-01-01,STX,TEXAS,EPG0,Natural Gas,VEU,Electric Power Consumption,N3045TX2,Texas Natural Gas Deliveries to Electric Power...,128086,MMCF
1,2023-02-01,STX,TEXAS,EPG0,Natural Gas,VEU,Electric Power Consumption,N3045TX2,Texas Natural Gas Deliveries to Electric Power...,118982,MMCF
2,2023-03-01,STX,TEXAS,EPG0,Natural Gas,VEU,Electric Power Consumption,N3045TX2,Texas Natural Gas Deliveries to Electric Power...,120711,MMCF
3,2023-04-01,STX,TEXAS,EPG0,Natural Gas,VEU,Electric Power Consumption,N3045TX2,Texas Natural Gas Deliveries to Electric Power...,119862,MMCF
4,2023-05-01,STX,TEXAS,EPG0,Natural Gas,VEU,Electric Power Consumption,N3045TX2,Texas Natural Gas Deliveries to Electric Power...,173024,MMCF
5,2023-06-01,STX,TEXAS,EPG0,Natural Gas,VEU,Electric Power Consumption,N3045TX2,Texas Natural Gas Deliveries to Electric Power...,210449,MMCF
6,2023-07-01,STX,TEXAS,EPG0,Natural Gas,VEU,Electric Power Consumption,N3045TX2,Texas Natural Gas Deliveries to Electric Power...,234904,MMCF
7,2023-08-01,STX,TEXAS,EPG0,Natural Gas,VEU,Electric Power Consumption,N3045TX2,Texas Natural Gas Deliveries to Electric Power...,268351,MMCF
8,2023-09-01,STX,TEXAS,EPG0,Natural Gas,VEU,Electric Power Consumption,N3045TX2,Texas Natural Gas Deliveries to Electric Power...,214662,MMCF
9,2023-10-01,STX,TEXAS,EPG0,Natural Gas,VEU,Electric Power Consumption,N3045TX2,Texas Natural Gas Deliveries to Electric Power...,152893,MMCF


In [19]:
# Multi-state annual consumption -- top producing states
df = get_consumption(
    state=["TX", "PA", "LA", "OK", "OH"],
    frequency="annual",
    start="2018",
    end="2023",
)
df.head(10)

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2018-01-01,SLA,USA-LA,EPG0,Natural Gas,VRS,Residential Consumption,N3010LA2,Louisiana Natural Gas Residential Consumption ...,37838,MMCF
1,2019-01-01,SLA,USA-LA,EPG0,Natural Gas,VRS,Residential Consumption,N3010LA2,Louisiana Natural Gas Residential Consumption ...,35935,MMCF
2,2020-01-01,SLA,USA-LA,EPG0,Natural Gas,VRS,Residential Consumption,N3010LA2,Louisiana Natural Gas Residential Consumption ...,32097,MMCF
3,2021-01-01,SLA,USA-LA,EPG0,Natural Gas,VRS,Residential Consumption,N3010LA2,Louisiana Natural Gas Residential Consumption ...,36270,MMCF
4,2022-01-01,SLA,USA-LA,EPG0,Natural Gas,VRS,Residential Consumption,N3010LA2,Louisiana Natural Gas Residential Consumption ...,35318,MMCF
5,2023-01-01,SLA,USA-LA,EPG0,Natural Gas,VRS,Residential Consumption,N3010LA2,Louisiana Natural Gas Residential Consumption ...,29494,MMCF
6,2018-01-01,SOH,OHIO,EPG0,Natural Gas,VRS,Residential Consumption,N3010OH2,Ohio Natural Gas Residential Consumption (MMcf),301232,MMCF
7,2019-01-01,SOH,OHIO,EPG0,Natural Gas,VRS,Residential Consumption,N3010OH2,Ohio Natural Gas Residential Consumption (MMcf),290080,MMCF
8,2020-01-01,SOH,OHIO,EPG0,Natural Gas,VRS,Residential Consumption,N3010OH2,Ohio Natural Gas Residential Consumption (MMcf),271863,MMCF
9,2021-01-01,SOH,OHIO,EPG0,Natural Gas,VRS,Residential Consumption,N3010OH2,Ohio Natural Gas Residential Consumption (MMcf),272676,MMCF


In [20]:
# Heat content sub-route for California
df = get_consumption(state="CA", route="heat", frequency="annual", start="2015")
df.head()

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2015-01-01,SCA,CALIFORNIA,EPG0,Natural Gas,VGTH,Heat Content Delivered to Consumers,NGA_EPG0_VGTH_SCA_BTUCF,California Heat Content of Natural Gas Deliver...,1036,BTU/CF
1,2016-01-01,SCA,CALIFORNIA,EPG0,Natural Gas,VGTH,Heat Content Delivered to Consumers,NGA_EPG0_VGTH_SCA_BTUCF,California Heat Content of Natural Gas Deliver...,1035,BTU/CF
2,2017-01-01,SCA,CALIFORNIA,EPG0,Natural Gas,VGTH,Heat Content Delivered to Consumers,NGA_EPG0_VGTH_SCA_BTUCF,California Heat Content of Natural Gas Deliver...,1035,BTU/CF
3,2018-01-01,SCA,CALIFORNIA,EPG0,Natural Gas,VGTH,Heat Content Delivered to Consumers,NGA_EPG0_VGTH_SCA_BTUCF,California Heat Content of Natural Gas Deliver...,1033,BTU/CF
4,2019-01-01,SCA,CALIFORNIA,EPG0,Natural Gas,VGTH,Heat Content Delivered to Consumers,NGA_EPG0_VGTH_SCA_BTUCF,California Heat Content of Natural Gas Deliver...,1034,BTU/CF


### 5b. Production

In [21]:
# Shale gas production in Pennsylvania & Texas
df = get_production(state=["PA", "TX"], route="shalegas", frequency="annual", start="2010")
df.head(10)

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2010-01-01,SPA,USA-PA,EPG0,Natural Gas,R5302,"Shale, Reserves Based Production",RES_EPG0_R5302_SPA_BCF,Pennsylvania Shale Production (Billion Cubic F...,396,BCF
1,2011-01-01,SPA,USA-PA,EPG0,Natural Gas,R5302,"Shale, Reserves Based Production",RES_EPG0_R5302_SPA_BCF,Pennsylvania Shale Production (Billion Cubic F...,1068,BCF
2,2012-01-01,SPA,USA-PA,EPG0,Natural Gas,R5302,"Shale, Reserves Based Production",RES_EPG0_R5302_SPA_BCF,Pennsylvania Shale Production (Billion Cubic F...,2036,BCF
3,2013-01-01,SPA,USA-PA,EPG0,Natural Gas,R5302,"Shale, Reserves Based Production",RES_EPG0_R5302_SPA_BCF,Pennsylvania Shale Production (Billion Cubic F...,3076,BCF
4,2014-01-01,SPA,USA-PA,EPG0,Natural Gas,R5302,"Shale, Reserves Based Production",RES_EPG0_R5302_SPA_BCF,Pennsylvania Shale Production (Billion Cubic F...,4009,BCF
5,2015-01-01,SPA,USA-PA,EPG0,Natural Gas,R5302,"Shale, Reserves Based Production",RES_EPG0_R5302_SPA_BCF,Pennsylvania Shale Production (Billion Cubic F...,4597,BCF
6,2016-01-01,SPA,USA-PA,EPG0,Natural Gas,R5302,"Shale, Reserves Based Production",RES_EPG0_R5302_SPA_BCF,Pennsylvania Shale Production (Billion Cubic F...,5049,BCF
7,2017-01-01,SPA,USA-PA,EPG0,Natural Gas,R5302,"Shale, Reserves Based Production",RES_EPG0_R5302_SPA_BCF,Pennsylvania Shale Production (Billion Cubic F...,5365,BCF
8,2018-01-01,SPA,USA-PA,EPG0,Natural Gas,R5302,"Shale, Reserves Based Production",RES_EPG0_R5302_SPA_BCF,Pennsylvania Shale Production (Billion Cubic F...,6079,BCF
9,2019-01-01,SPA,USA-PA,EPG0,Natural Gas,R5302,"Shale, Reserves Based Production",RES_EPG0_R5302_SPA_BCF,Pennsylvania Shale Production (Billion Cubic F...,6782,BCF


In [22]:
# Offshore production + coalbed methane in West Virginia
df_off = get_production(route="off", frequency="annual", start="2015")
df_cb = get_production(state="WV", route="coalbed", frequency="annual", start="2010")

print(f"Offshore rows: {len(df_off)}, WV coalbed rows: {len(df_cb)}")
df_off.head()

Offshore rows: 421, WV coalbed rows: 10


,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2015-01-01,R5F,NA,EPG0,Natural Gas,FGG,Withdrawals from Gas Wells,NA1000_R5F_2,Federal Offshore California Natural Gas Withdr...,6145,MMCF
1,2016-01-01,R5F,NA,EPG0,Natural Gas,FGG,Withdrawals from Gas Wells,NA1000_R5F_2,Federal Offshore California Natural Gas Withdr...,896,MMCF
2,2017-01-01,R5F,NA,EPG0,Natural Gas,FGG,Withdrawals from Gas Wells,NA1000_R5F_2,Federal Offshore California Natural Gas Withdr...,804,MMCF
3,2018-01-01,R5F,NA,EPG0,Natural Gas,FGG,Withdrawals from Gas Wells,NA1000_R5F_2,Federal Offshore California Natural Gas Withdr...,757,MMCF
4,2019-01-01,R5F,NA,EPG0,Natural Gas,FGG,Withdrawals from Gas Wells,NA1000_R5F_2,Federal Offshore California Natural Gas Withdr...,627,MMCF


### 5c. Movements

In [23]:
# Interstate pipeline flows for Louisiana
df = get_movements(state="LA", frequency="annual", start="2018")
df.head(10)

""


In [24]:
# Gas imports by country + exports by country
df_imp = get_movements(route="impc", frequency="annual", start="2015")
df_exp = get_movements(route="expc", frequency="annual", start="2018")

print(f"Import rows: {len(df_imp)}, Export rows: {len(df_exp)}")
df_imp.head()

Import rows: 562, Export rows: 1111


,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2015-01-01,NUS-Z00,U.S.,EPG0,Natural Gas,IM0,Imports,N9100US2,U.S. Natural Gas Imports (MMcf),2718094.0,MMCF
1,2016-01-01,NUS-Z00,U.S.,EPG0,Natural Gas,IM0,Imports,N9100US2,U.S. Natural Gas Imports (MMcf),3006497.0,MMCF
2,2017-01-01,NUS-Z00,U.S.,EPG0,Natural Gas,IM0,Imports,N9100US2,U.S. Natural Gas Imports (MMcf),3033226.0,MMCF
3,2018-01-01,NUS-Z00,U.S.,EPG0,Natural Gas,IM0,Imports,N9100US2,U.S. Natural Gas Imports (MMcf),2888847.0,MMCF
4,2019-01-01,NUS-Z00,U.S.,EPG0,Natural Gas,IM0,Imports,N9100US2,U.S. Natural Gas Imports (MMcf),2741717.0,MMCF


### 5d. Storage

In [25]:
# Weekly storage levels by region
for region in ["east", "midwest", "south_central", "mountain", "pacific"]:
    df = get_storage(region=region, frequency="weekly", start="2024-01", end="2024-03")
    print(f"  {region:15s}: {len(df)} rows")

  east           : 9 rows
  midwest        : 9 rows
  south_central  : 27 rows
  mountain       : 9 rows
  pacific        : 9 rows


In [26]:
# Multiple regions at once + LNG storage sub-route
df = get_storage(region=["east", "midwest"], frequency="weekly", start="2024-01", end="2024-03")
print(f"East & Midwest: {len(df)} rows")
df.head()

East & Midwest: 18 rows


,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2024-01-05,R32,NA,EPG0,Natural Gas,SWO,Underground Storage - Working Gas,NW2_EPG0_SWO_R32_BCF,Weekly Midwest Region Natural Gas Working Unde...,924,BCF
1,2024-01-12,R32,NA,EPG0,Natural Gas,SWO,Underground Storage - Working Gas,NW2_EPG0_SWO_R32_BCF,Weekly Midwest Region Natural Gas Working Unde...,873,BCF
2,2024-01-19,R32,NA,EPG0,Natural Gas,SWO,Underground Storage - Working Gas,NW2_EPG0_SWO_R32_BCF,Weekly Midwest Region Natural Gas Working Unde...,788,BCF
3,2024-01-26,R32,NA,EPG0,Natural Gas,SWO,Underground Storage - Working Gas,NW2_EPG0_SWO_R32_BCF,Weekly Midwest Region Natural Gas Working Unde...,727,BCF
4,2024-02-02,R32,NA,EPG0,Natural Gas,SWO,Underground Storage - Working Gas,NW2_EPG0_SWO_R32_BCF,Weekly Midwest Region Natural Gas Working Unde...,689,BCF


In [27]:
# LNG storage (annual only for this sub-route)
df = get_storage(route="lng", frequency="annual", start="2018")
df.head()

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2018-01-01,NUS,U.S.,EPG0,Natural Gas,SAL,LNG Storage Net Withdrawals,NA1350_NUS_2,U.S. Natural Gas LNG Storage Net Withdrawals (...,9248.0,MMCF
1,2019-01-01,NUS,U.S.,EPG0,Natural Gas,SAL,LNG Storage Net Withdrawals,NA1350_NUS_2,U.S. Natural Gas LNG Storage Net Withdrawals (...,3715.0,MMCF
2,2020-01-01,NUS,U.S.,EPG0,Natural Gas,SAL,LNG Storage Net Withdrawals,NA1350_NUS_2,U.S. Natural Gas LNG Storage Net Withdrawals (...,1718.0,MMCF
3,2021-01-01,NUS,U.S.,EPG0,Natural Gas,SAL,LNG Storage Net Withdrawals,NA1350_NUS_2,U.S. Natural Gas LNG Storage Net Withdrawals (...,156.0,MMCF
4,2022-01-01,NUS,U.S.,EPG0,Natural Gas,SAL,LNG Storage Net Withdrawals,NA1350_NUS_2,U.S. Natural Gas LNG Storage Net Withdrawals (...,-3527.0,MMCF


### 5e. Prices

In [28]:
# National gas prices, annual
df = get_prices(area="national", frequency="annual", start="2010")
df.head(10)

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2010-01-01,NUS,U.S.,EPG0,Natural Gas,PRS,Price Delivered to Residential Consumers,N3010US3,U.S. Price of Natural Gas Delivered to Residen...,11.39,$/MCF
1,2011-01-01,NUS,U.S.,EPG0,Natural Gas,PRS,Price Delivered to Residential Consumers,N3010US3,U.S. Price of Natural Gas Delivered to Residen...,11.03,$/MCF
2,2012-01-01,NUS,U.S.,EPG0,Natural Gas,PRS,Price Delivered to Residential Consumers,N3010US3,U.S. Price of Natural Gas Delivered to Residen...,10.65,$/MCF
3,2013-01-01,NUS,U.S.,EPG0,Natural Gas,PRS,Price Delivered to Residential Consumers,N3010US3,U.S. Price of Natural Gas Delivered to Residen...,10.32,$/MCF
4,2014-01-01,NUS,U.S.,EPG0,Natural Gas,PRS,Price Delivered to Residential Consumers,N3010US3,U.S. Price of Natural Gas Delivered to Residen...,10.97,$/MCF
5,2015-01-01,NUS,U.S.,EPG0,Natural Gas,PRS,Price Delivered to Residential Consumers,N3010US3,U.S. Price of Natural Gas Delivered to Residen...,10.38,$/MCF
6,2016-01-01,NUS,U.S.,EPG0,Natural Gas,PRS,Price Delivered to Residential Consumers,N3010US3,U.S. Price of Natural Gas Delivered to Residen...,10.05,$/MCF
7,2017-01-01,NUS,U.S.,EPG0,Natural Gas,PRS,Price Delivered to Residential Consumers,N3010US3,U.S. Price of Natural Gas Delivered to Residen...,10.91,$/MCF
8,2018-01-01,NUS,U.S.,EPG0,Natural Gas,PRS,Price Delivered to Residential Consumers,N3010US3,U.S. Price of Natural Gas Delivered to Residen...,10.50,$/MCF
9,2019-01-01,NUS,U.S.,EPG0,Natural Gas,PRS,Price Delivered to Residential Consumers,N3010US3,U.S. Price of Natural Gas Delivered to Residen...,10.51,$/MCF


In [29]:
# State-level prices for TX & CA + futures prices + residential/commercial in NY
df_state = get_prices(area=["TX", "CA"], frequency="monthly", start="2023-01", end="2024-06")
df_fut = get_prices(route="fut", frequency="monthly", start="2023-01")
df_rescom = get_prices(area="NY", route="rescom", frequency="monthly", start="2023-01")

print(f"State prices: {len(df_state)} rows")
print(f"Futures:      {len(df_fut)} rows")
print(f"NY res/com:   {len(df_rescom)} rows")
df_fut.head()

State prices: 288 rows
Futures:      140 rows
NY res/com:   74 rows


,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2023-01-01,Y35NY,NEW YORK CITY,EPG0,Natural Gas,PE1,Future Contract 1,RNGC1,Natural Gas Futures Contract 1 (Dollars per Mi...,3.42,$/MMBTU
1,2023-02-01,Y35NY,NEW YORK CITY,EPG0,Natural Gas,PE1,Future Contract 1,RNGC1,Natural Gas Futures Contract 1 (Dollars per Mi...,2.44,$/MMBTU
2,2023-03-01,Y35NY,NEW YORK CITY,EPG0,Natural Gas,PE1,Future Contract 1,RNGC1,Natural Gas Futures Contract 1 (Dollars per Mi...,2.41,$/MMBTU
3,2023-04-01,Y35NY,NEW YORK CITY,EPG0,Natural Gas,PE1,Future Contract 1,RNGC1,Natural Gas Futures Contract 1 (Dollars per Mi...,2.20,$/MMBTU
4,2023-05-01,Y35NY,NEW YORK CITY,EPG0,Natural Gas,PE1,Future Contract 1,RNGC1,Natural Gas Futures Contract 1 (Dollars per Mi...,2.30,$/MMBTU


### 5f. Exploration & Reserves

In [30]:
# Shale gas reserves + TX drilling activity + well costs
df_shale = get_exploration(route="shalegas", frequency="annual", start="2010")
df_drill = get_exploration(state="TX", route="drill", frequency="annual", start="2005")
df_cost = get_exploration(route="wellcost", frequency="annual", start="2005")

print(f"Shale reserves: {len(df_shale)} rows")
print(f"TX drilling:    {len(df_drill)} rows")
print(f"Well costs:     {len(df_cost)} rows")
df_shale.head()

Shale reserves: 4813 rows
TX drilling:    0 rows
Well costs:     30 rows


,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2010-01-01,SAR,USA-AR,EPG0,Natural Gas,R5301,"Shale, Proved Reserves",RES_EPG0_R5301_SAR_BCF,Arkansas Shale Proved Reserves (Billion Cubic ...,12526.0,BCF
1,2011-01-01,SAR,USA-AR,EPG0,Natural Gas,R5301,"Shale, Proved Reserves",RES_EPG0_R5301_SAR_BCF,Arkansas Shale Proved Reserves (Billion Cubic ...,14808.0,BCF
2,2012-01-01,SAR,USA-AR,EPG0,Natural Gas,R5301,"Shale, Proved Reserves",RES_EPG0_R5301_SAR_BCF,Arkansas Shale Proved Reserves (Billion Cubic ...,9779.0,BCF
3,2013-01-01,SAR,USA-AR,EPG0,Natural Gas,R5301,"Shale, Proved Reserves",RES_EPG0_R5301_SAR_BCF,Arkansas Shale Proved Reserves (Billion Cubic ...,12231.0,BCF
4,2014-01-01,SAR,USA-AR,EPG0,Natural Gas,R5301,"Shale, Proved Reserves",RES_EPG0_R5301_SAR_BCF,Arkansas Shale Proved Reserves (Billion Cubic ...,11695.0,BCF


### 5g. Summary

In [31]:
# Monthly supply & disposition
df = get_summary(route="sndm", frequency="monthly", start="2023-01")
df.head(10)

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2023-01-01,NUS,U.S.,EPG0,Natural Gas,FGW,Gross Withdrawals,N9010US1,U.S. Natural Gas Gross Withdrawals (Bcf),3841,BCF
1,2023-02-01,NUS,U.S.,EPG0,Natural Gas,FGW,Gross Withdrawals,N9010US1,U.S. Natural Gas Gross Withdrawals (Bcf),3456,BCF
2,2023-03-01,NUS,U.S.,EPG0,Natural Gas,FGW,Gross Withdrawals,N9010US1,U.S. Natural Gas Gross Withdrawals (Bcf),3857,BCF
3,2023-04-01,NUS,U.S.,EPG0,Natural Gas,FGW,Gross Withdrawals,N9010US1,U.S. Natural Gas Gross Withdrawals (Bcf),3718,BCF
4,2023-05-01,NUS,U.S.,EPG0,Natural Gas,FGW,Gross Withdrawals,N9010US1,U.S. Natural Gas Gross Withdrawals (Bcf),3859,BCF
5,2023-06-01,NUS,U.S.,EPG0,Natural Gas,FGW,Gross Withdrawals,N9010US1,U.S. Natural Gas Gross Withdrawals (Bcf),3688,BCF
6,2023-07-01,NUS,U.S.,EPG0,Natural Gas,FGW,Gross Withdrawals,N9010US1,U.S. Natural Gas Gross Withdrawals (Bcf),3800,BCF
7,2023-08-01,NUS,U.S.,EPG0,Natural Gas,FGW,Gross Withdrawals,N9010US1,U.S. Natural Gas Gross Withdrawals (Bcf),3807,BCF
8,2023-09-01,NUS,U.S.,EPG0,Natural Gas,FGW,Gross Withdrawals,N9010US1,U.S. Natural Gas Gross Withdrawals (Bcf),3723,BCF
9,2023-10-01,NUS,U.S.,EPG0,Natural Gas,FGW,Gross Withdrawals,N9010US1,U.S. Natural Gas Gross Withdrawals (Bcf),3876,BCF


---

## 6. Nuclear Outages

In [32]:
# Daily US-wide nuclear outage data
df = get_data(
    "nuclear-outages", "us-nuclear-outages",
    data=["capacity", "outage", "percentOutage"],
    frequency="daily",
    start="2024-01-01",
    end="2024-01-31",
)
df.head(10)

,period,capacity,outage,percentOutage,capacity-units,outage-units,percentOutage-units
0,2024-01-31,98015.8,7161.583,7.31,megawatts,megawatts,percent
1,2024-01-30,98015.8,8282.865,8.45,megawatts,megawatts,percent
2,2024-01-29,98015.8,7284.546,7.43,megawatts,megawatts,percent
3,2024-01-28,98015.8,8103.759,8.27,megawatts,megawatts,percent
4,2024-01-27,98015.8,5890.067,6.01,megawatts,megawatts,percent
5,2024-01-26,98015.8,5772.292,5.89,megawatts,megawatts,percent
6,2024-01-25,98015.8,5397.448,5.51,megawatts,megawatts,percent
7,2024-01-24,98015.8,4322.092,4.41,megawatts,megawatts,percent
8,2024-01-23,98015.8,4608.183,4.7,megawatts,megawatts,percent
9,2024-01-22,98015.8,4749.747,4.85,megawatts,megawatts,percent


---

## 7. Total Energy

In [33]:
# Monthly primary energy production: petroleum, natural gas, coal
df = get_data(
    "total-energy",
    facets={"msn": ["PAPRB", "NGPRB", "CLPRB"]},
    frequency="monthly",
    start="2023-01",
    end="2024-06",
)
df.head(10)

""


In [34]:
# Annual total energy consumption (2000-2023)
df = get_data(
    "total-energy",
    facets={"msn": ["TETCB"]},
    frequency="annual",
    start="2000",
    end="2023",
)
df.head(10)

""


---

## 8. SEDS -- State Energy Data System

In [35]:
# Annual total energy consumption by state
df = get_data(
    "seds",
    facets={"seriesId": "TETCB", "stateId": ["TX", "CA", "NY", "FL", "WY"]},
    frequency="annual",
    start="2010",
    end="2022",
)
df.head(10)

,period,seriesId,seriesDescription,stateId,stateDescription,value,unit
0,2022-01-01,TETCB,Total energy consumption,CA,California,6855276,Billion Btu
1,2022-01-01,TETCB,Total energy consumption,FL,Florida,4335139,Billion Btu
2,2022-01-01,TETCB,Total energy consumption,NY,New York,3455074,Billion Btu
3,2022-01-01,TETCB,Total energy consumption,TX,Texas,13831344,Billion Btu
4,2022-01-01,TETCB,Total energy consumption,WY,Wyoming,494352,Billion Btu
5,2021-01-01,TETCB,Total energy consumption,CA,California,6778817,Billion Btu
6,2021-01-01,TETCB,Total energy consumption,FL,Florida,4257181,Billion Btu
7,2021-01-01,TETCB,Total energy consumption,NY,New York,3345022,Billion Btu
8,2021-01-01,TETCB,Total energy consumption,TX,Texas,13721691,Billion Btu
9,2021-01-01,TETCB,Total energy consumption,WY,Wyoming,481378,Billion Btu


---

## 9. STEO -- Short-Term Energy Outlook

In [36]:
# Monthly forecast of WTI crude oil price
df = get_data(
    "steo",
    facets={"seriesId": "WTIPUUS"},
    frequency="monthly",
    start="2024-01",
)
df.head(10)

,period,seriesId,seriesDescription,value,unit
0,2027-12-01,WTIPUUS,West Texas Intermediate Crude Oil Price,64.0,dollars per barrel
1,2027-11-01,WTIPUUS,West Texas Intermediate Crude Oil Price,66.0,dollars per barrel
2,2027-10-01,WTIPUUS,West Texas Intermediate Crude Oil Price,68.0,dollars per barrel
3,2027-09-01,WTIPUUS,West Texas Intermediate Crude Oil Price,70.0,dollars per barrel
4,2027-08-01,WTIPUUS,West Texas Intermediate Crude Oil Price,72.0,dollars per barrel
5,2027-07-01,WTIPUUS,West Texas Intermediate Crude Oil Price,73.0,dollars per barrel
6,2027-06-01,WTIPUUS,West Texas Intermediate Crude Oil Price,74.0,dollars per barrel
7,2027-05-01,WTIPUUS,West Texas Intermediate Crude Oil Price,75.0,dollars per barrel
8,2027-04-01,WTIPUUS,West Texas Intermediate Crude Oil Price,75.0,dollars per barrel
9,2027-03-01,WTIPUUS,West Texas Intermediate Crude Oil Price,76.0,dollars per barrel


In [37]:
# Quarterly natural gas production forecast
df = get_data(
    "steo",
    facets={"seriesId": "NGPRPUS"},
    frequency="quarterly",
    start="2024-01",
)
df.head()

/Users/egutierrez/Documents/code/pygasflow/eiapy/client.py:122: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["period"] = pd.to_datetime(df["period"])


,period,seriesId,seriesDescription,value,unit
0,2027-10-01,NGPRPUS,Natural Gas Total Dry Production,114.279866,billion cubic feet per day
1,2027-07-01,NGPRPUS,Natural Gas Total Dry Production,113.032521,billion cubic feet per day
2,2027-04-01,NGPRPUS,Natural Gas Total Dry Production,112.082182,billion cubic feet per day
3,2027-01-01,NGPRPUS,Natural Gas Total Dry Production,110.944552,billion cubic feet per day
4,2026-10-01,NGPRPUS,Natural Gas Total Dry Production,110.334848,billion cubic feet per day


---

## 10. International

In [38]:
# Annual crude oil production for top 5 producers
df = get_data(
    "international",
    facets={
        "productId": "57",
        "activityId": "1",
        "countryRegionId": ["USA", "SAU", "RUS", "CAN", "IRQ"],
        "unit": "TBPD",
    },
    frequency="annual",
    start="2015",
    end="2023",
)
df.head(10)

,period,productId,productName,activityId,activityName,countryRegionId,countryRegionName,countryRegionTypeId,countryRegionTypeName,dataFlagId,dataFlagDescription,unitName,value,unit
0,2023-01-01,57,Crude oil including lease condensate,1,Production,CAN,Canada,c,Country,None,None,thousand barrels per day,4594.118260,TBPD
1,2023-01-01,57,Crude oil including lease condensate,1,Production,IRQ,Iraq,c,Country,None,None,thousand barrels per day,4353.164384,TBPD
2,2023-01-01,57,Crude oil including lease condensate,1,Production,RUS,Russia,c,Country,None,None,thousand barrels per day,10276.559342,TBPD
3,2023-01-01,57,Crude oil including lease condensate,1,Production,SAU,Saudi Arabia,c,Country,None,None,thousand barrels per day,9748.442518,TBPD
4,2023-01-01,57,Crude oil including lease condensate,1,Production,USA,United States,c,Country,None,None,thousand barrels per day,12943.383537,TBPD
5,2022-01-01,57,Crude oil including lease condensate,1,Production,CAN,Canada,c,Country,None,None,thousand barrels per day,4543.292121,TBPD
6,2022-01-01,57,Crude oil including lease condensate,1,Production,IRQ,Iraq,c,Country,None,None,thousand barrels per day,4470.506849,TBPD
7,2022-01-01,57,Crude oil including lease condensate,1,Production,RUS,Russia,c,Country,None,None,thousand barrels per day,10318.971480,TBPD
8,2022-01-01,57,Crude oil including lease condensate,1,Production,SAU,Saudi Arabia,c,Country,None,None,thousand barrels per day,10664.561644,TBPD
9,2022-01-01,57,Crude oil including lease condensate,1,Production,USA,United States,c,Country,None,None,thousand barrels per day,12004.217773,TBPD


---

## 11. Densified Biomass

In [39]:
# Monthly wood pellet production by region
df = get_data(
    "densified-biomass", "production-by-region",
    data=["production"],
    frequency="monthly",
    start="2023-01",
)
df.head(10)

,period,fuelTypeId,fuelTypeDescription,region,production,production-units
0,2025-10-01,CBL,Compressed Bricks Log,EAST,0,tons
1,2025-10-01,CBL,Compressed Bricks Log,SOUTH,0,tons
2,2025-10-01,CBL,Compressed Bricks Log,US-TOTAL,0,tons
3,2025-10-01,CBL,Compressed Bricks Log,WEST,0,tons
4,2025-10-01,WPPS,Wood Pellets Premium Standard,EAST,100543,tons
5,2025-10-01,WPPS,Wood Pellets Premium Standard,SOUTH,25222,tons
6,2025-10-01,WPPS,Wood Pellets Premium Standard,US-TOTAL,155744,tons
7,2025-10-01,WPPS,Wood Pellets Premium Standard,WEST,29979,tons
8,2025-10-01,WPU,Wood Pellets Utility,EAST,0,tons
9,2025-10-01,WPU,Wood Pellets Utility,SOUTH,860423,tons


In [40]:
# Biomass export sales & pricing
df = get_data(
    "densified-biomass", "export-sales-and-price",
    frequency="monthly",
    start="2023-01",
)
df.head()

,period,quantity,average-price,quantity-units,average-price-units
0,2025-10-01,773661,203.18,tons,USD per ton
1,2025-09-01,762453,212.82,tons,USD per ton
2,2025-08-01,785142,207.09,tons,USD per ton
3,2025-07-01,880965,205.83,tons,USD per ton
4,2025-06-01,717018,207.73,tons,USD per ton


---

## 12. AEO -- Annual Energy Outlook

In [42]:
# AEO 2025 projections through 2050
df = get_data(
    "aeo", "2025",
    frequency="annual",
    start="2028",
    end="2029",
)
df.head(10)

KeyboardInterrupt: 

---

## 13. Raw Route -- Bypassing the Registry

You can always pass a raw EIA route string directly via `route=`, useful for endpoints not yet in the bundled registry.

In [43]:
# Petroleum distillate prices via raw route string
df = get_data(
    "petroleum",
    route="petroleum/pri/dist",
    frequency="monthly",
    start="2024-01",
    end="2024-06",
)
df.head(10)

""


---

## 14. Metadata Inspection

Use `get_metadata()` to discover available facets, frequencies, and data columns for any endpoint before querying.

In [44]:
# Inspect metadata across different categories
endpoints = [
    ("electricity",      "retail-sales",            None),
    ("petroleum",        "pri",                     "allmg"),
    ("coal",             "consumption-and-quality",  None),
    ("crude-oil-imports", None,                      None),
    ("nuclear-outages",  "us-nuclear-outages",       None),
]

for cat, group, variant in endpoints:
    label = f"{cat}/{group or '*'}/{variant or 'default'}"
    meta = get_metadata(cat, group, variant)
    freqs = [f["id"] for f in meta.get("frequency", [])]
    facets = [f["id"] for f in meta.get("facets", [])]
    data_cols = list(meta.get("data", {}).keys())
    print(f"{label}:")
    print(f"  frequencies: {freqs}")
    print(f"  facets:      {facets}")
    print(f"  data:        {data_cols}")
    print()

electricity/retail-sales/default:
  frequencies: ['monthly', 'quarterly', 'annual']
  facets:      ['stateid', 'sectorid']
  data:        ['revenue', 'sales', 'price', 'customers']

petroleum/pri/allmg:
  frequencies: ['monthly', 'annual']
  facets:      ['duoarea', 'product', 'process', 'series']
  data:        ['value']

coal/consumption-and-quality/default:
  frequencies: ['quarterly', 'annual']
  facets:      ['location', 'sector']
  data:        ['receipts', 'consumption', 'stocks', 'price', 'heat-content', 'sulfur-content', 'ash-content']

crude-oil-imports/*/default:
  frequencies: ['monthly', 'annual']
  facets:      ['originId', 'originType', 'destinationId', 'destinationType', 'gradeId']
  data:        ['quantity']

nuclear-outages/us-nuclear-outages/default:
  frequencies: ['daily']
  facets:      []
  data:        ['capacity', 'outage', 'percentOutage']



In [45]:
# Natural gas metadata via legacy helper
meta = get_available_series("consumption")
print("Natural gas consumption metadata:")
print(f"  frequencies: {[f['id'] for f in meta.get('frequency', [])]}")
print(f"  facets:      {[f['id'] for f in meta.get('facets', [])]}")
print(f"  data:        {list(meta.get('data', {}).keys())}")

Natural gas consumption metadata:
  frequencies: ['monthly', 'annual']
  facets:      ['duoarea', 'product', 'process', 'series']
  data:        ['value']


---

## 15. Error Handling

Validation errors are caught locally (no API call needed). API errors are raised as typed exceptions.

In [46]:
from eiapy import (
    EIAError,
    MissingAPIKeyError,
    AuthenticationError,
    RateLimitError,
    NotFoundError,
    RequestFailedError,
)

# Bad category
try:
    get_data("fake-category", "group")
except ValueError as e:
    print(f"Bad category:  {e}")

# Bad frequency
try:
    get_data("electricity", "retail-sales", frequency="biweekly")
except ValueError as e:
    print(f"Bad frequency: {e}")

# Bad date format
try:
    get_data("electricity", "retail-sales", start="not-a-date")
except ValueError as e:
    print(f"Bad date:      {e}")

# Bad storage region
try:
    get_storage(region="atlantis")
except ValueError as e:
    print(f"Bad region:    {e}")

# Missing group for multi-group category
try:
    get_data("natural-gas")
except ValueError as e:
    print(f"No group:      {e}")

Bad category:  Unknown category 'fake-category'. Valid categories: ['aeo', 'coal', 'crude-oil-imports', 'densified-biomass', 'electricity', 'ieo', 'international', 'natural-gas', 'nuclear-outages', 'petroleum', 'seds', 'steo', 'total-energy'].
Bad frequency: Invalid frequency 'biweekly'. Must be one of: ['annual', 'daily', 'monthly', 'quarterly', 'weekly'].
Bad date:      Invalid start='not-a-date'. Expected YYYY, YYYY-MM or YYYY-MM-DD.
Bad region:    Unknown storage region 'atlantis'. Valid: ['east', 'lower48', 'midwest', 'mountain', 'national', 'pacific', 'south_central'].
No group:      Category 'natural-gas' has multiple groups: ['cons', 'enr', 'move', 'pri', 'prod', 'stor', 'sum']. Please specify a group.
